Mount Google Drive & Create Folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data"

RAW_DIR = f"{BASE_DIR}/raw_pdfs"
PROCESSED_DIR = f"{BASE_DIR}/processed"
CORPUS_DIR = f"{BASE_DIR}/sri_lanka_legal_corpus"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CORPUS_DIR, exist_ok=True)

print("Folder structure created.")

Mounted at /content/drive
Folder structure created.


In [7]:
import zipfile
import shutil

TEMP_UNZIP_DIR = f"{BASE_DIR}/temp_unzipped_pdfs"
os.makedirs(TEMP_UNZIP_DIR, exist_ok=True)

zip_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith(".zip")]

if zip_files:
    print(f"Found {len(zip_files)} zip files. Extracting...")
    for zip_file_name in zip_files:
        zip_file_path = os.path.join(RAW_DIR, zip_file_name)
        try:
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                # Extract into a subfolder within TEMP_UNZIP_DIR to avoid name clashes
                extraction_folder = os.path.join(TEMP_UNZIP_DIR, os.path.splitext(zip_file_name)[0])
                os.makedirs(extraction_folder, exist_ok=True)
                zip_ref.extractall(extraction_folder)
            print(f"Extracted '{zip_file_name}' to '{extraction_folder}'")
        except zipfile.BadZipFile:
            print(f"Warning: '{zip_file_name}' is not a valid zip file. Skipping.")
        except Exception as e:
            print(f"Error extracting '{zip_file_name}': {e}")
else:
    print("No zip files found in RAW_DIR.")


Found 41 zip files. Extracting...
Extracted 'Sri-Lanka-Law-Report-1982--01146027300.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-Law-Report-1982--01146027300'
Extracted 'Sri-Lanka-Law-Report-1981--01152412000.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-Law-Report-1981--01152412000'
Extracted 'Sri-Lanka-Law-Report-1983--01135193700.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-Law-Report-1983--01135193700'
Extracted 'Sri-Lanka-Law-Report-1984--01112298900.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-Law-Report-1984--01112298900'
Extracted 'Sri-Lanka-Law-Report-1989--01147314000.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-Law-Report-1989--01147314000'
Extracted 'Sri-Lanka-Law-Report-1986--01108935900.zip' to '/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs/Sri-Lanka-

In [8]:
pip install pymupdf pdfplumber pytesseract pdf2image pillow nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 93.2 MB/s eta 0:00:00


In [9]:
import os
import json
import re
import shutil
import nltk
from datetime import datetime

# Download NLTK tokenizers
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Enhanced OCR/text extraction function
def robust_extract_text_from_pdf(file_path):
    """
    Extract text from PDF using multiple methods with fallbacks
    """
    text = ""

    # Method 1: Try PyMuPDF (fitz) first - fastest for text-based PDFs
    try:
        import fitz
        doc = fitz.open(file_path)
        for page_num, page in enumerate(doc):
            page_text = page.get_text()
            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n{page_text}"
        doc.close()
        if text.strip():
            print(f"  ✓ Text extracted via PyMuPDF ({len(text)} chars)")
            return text
    except Exception as e:
        print(f"  ⚠️  PyMuPDF failed: {str(e)}")

    # Method 2: Try pdfplumber
    try:
        import pdfplumber
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += f"\n--- Page {page_num + 1} ---\n{page_text}"
        if text.strip():
            print(f"  ✓ Text extracted via pdfplumber ({len(text)} chars)")
            return text
    except Exception as e:
        print(f"  ⚠️  pdfplumber failed: {str(e)}")

    # Method 3: Try OCR with pytesseract for scanned PDFs
    try:
        from pdf2image import convert_from_path
        import pytesseract

        print(f"  🔍 Attempting OCR conversion...")
        images = convert_from_path(file_path, dpi=200, first_page=1, last_page=3)  # Limit pages for testing

        for page_num, image in enumerate(images):
            page_text = pytesseract.image_to_string(image)
            if page_text:
                text += f"\n--- Page {page_num + 1} (OCR) ---\n{page_text}"

        if text.strip():
            print(f"  ✓ Text extracted via OCR ({len(text)} chars)")
            return text
        else:
            print(f"  ⚠️  OCR produced no text")

    except Exception as e:
        print(f"  ⚠️  OCR failed: {str(e)}")

    return ""

# PDF validation function
def validate_pdf(file_path):
    """Check if file is a valid PDF"""
    try:
        with open(file_path, 'rb') as f:
            header = f.read(4)
            return header == b'%PDF'
    except:
        return False

# Text cleaning function
def clean_text(text):
    """Clean extracted text"""
    if not text:
        return ""

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove page numbers and headers/footers (basic)
    text = re.sub(r'\n\d+\n', '\n', text)

    return text.strip()

# Sentence segmentation
def segment_sentences(text):
    """Segment text into sentences"""
    if not text:
        return []

    try:
        from nltk.tokenize import sent_tokenize
        sentences = sent_tokenize(text)
        return [s.strip() for s in sentences if s.strip()]
    except:
        # Fallback: simple regex-based sentence splitting
        sentences = re.split(r'[.!?]+', text)
        return [s.strip() for s in sentences if s.strip()]

# Metadata extraction functions
def extract_case_year(text):
    """Extract case year from text"""
    years = re.findall(r'\b(19|20)\d{2}\b', text)
    if years:
        return int(years[0])
    return None

def extract_case_number(text):
    """Extract case number from text"""
    # Common Sri Lanka case number patterns
    patterns = [
        r'Case\s+No\.?\s*[:\-]?\s*([A-Za-z0-9/\-]+)',
        r'No\.?\s*(\d+[\-/]\d+)',
        r'SC\/?([A-Za-z]+\s*\d+\s*\/\s*\d+)',
        r'CA\s+(\d+)\s*\/\s*\d+'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return None

def extract_court(text):
    """Extract court information"""
    courts = ['Supreme Court', 'Court of Appeal', 'High Court', 'District Court', 'Magistrate Court']
    for court in courts:
        if court.lower() in text.lower():
            return court
    return "Unknown"

# Define your directories (update these paths)
RAW_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/raw_pdfs"
TEMP_UNZIP_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs"
PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons"

# Create processed directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Get all PDF files
print("🔍 Scanning for PDF files...")
raw_dir_pdfs = [os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if f.lower().endswith(".pdf")]
unzipped_pdfs = []

if os.path.exists(TEMP_UNZIP_DIR):
    for root, _, files in os.walk(TEMP_UNZIP_DIR):
        for file in files:
            if file.lower().endswith(".pdf"):
                unzipped_pdfs.append(os.path.join(root, file))

all_pdf_paths = raw_dir_pdfs + unzipped_pdfs
print(f"📁 Found {len(all_pdf_paths)} PDF files for processing")

# Process files
dataset = []
failed_pdfs = []
success_count = 0

print(f"\n🚀 Starting PDF processing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for i, file_path in enumerate(all_pdf_paths):
    file_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(all_pdf_paths)}] Processing: {file_name}")

    # Validate PDF
    if not validate_pdf(file_path):
        print(f"  ❌ Invalid PDF file")
        failed_pdfs.append(file_name)
        continue

    # Check file size
    file_size = os.path.getsize(file_path)
    if file_size < 1000:  # Less than 1KB
        print(f"  ⚠️  File too small ({file_size} bytes), likely corrupted")
        failed_pdfs.append(file_name)
        continue

    try:
        # Extract text
        raw_text = robust_extract_text_from_pdf(file_path)

        if not raw_text or len(raw_text.strip()) < 100:  # Minimum text threshold
            print(f"  ❌ Insufficient text extracted ({len(raw_text.strip())} chars)")
            failed_pdfs.append(file_name)
            continue

        # Process text
        cleaned_text = clean_text(raw_text)
        sentences = segment_sentences(cleaned_text)

        # Extract metadata
        case_year = extract_case_year(cleaned_text)
        case_number = extract_case_number(cleaned_text)
        court = extract_court(cleaned_text)

        # Create entry
        entry = {
            "file_name": file_name,
            "file_size_bytes": file_size,
            "raw_text": raw_text,
            "cleaned_text": cleaned_text,
            "sentences": sentences,
            "case_year": case_year,
            "case_number": case_number,
            "court": court,
            "processing_timestamp": datetime.now().isoformat(),
            "text_length": len(cleaned_text),
            "sentence_count": len(sentences)
        }

        # Save JSON
        json_filename = file_name.replace(".pdf", ".json")
        save_path = os.path.join(PROCESSED_DIR, json_filename)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(entry, f, indent=2, ensure_ascii=False)

        dataset.append(entry)
        success_count += 1
        print(f"  ✅ Successfully processed and saved: {json_filename}")
        print(f"  📊 Stats: {len(cleaned_text)} chars, {len(sentences)} sentences, Year: {case_year}, Court: {court}")

    except Exception as e:
        print(f"  ❌ Error processing {file_name}: {str(e)}")
        failed_pdfs.append(file_name)

# Print summary
print(f"\n{'='*50}")
print("📊 PROCESSING SUMMARY")
print(f"{'='*50}")
print(f"✅ Successfully processed: {success_count}/{len(all_pdf_paths)}")
print(f"❌ Failed: {len(failed_pdfs)}")
print(f"📈 Success rate: {(success_count/len(all_pdf_paths))*100:.1f}%")

if failed_pdfs:
    print(f"\n❌ Failed files ({len(failed_pdfs)}):")
    for failed in failed_pdfs[:10]:  # Show first 10 failed files
        print(f"  - {failed}")
    if len(failed_pdfs) > 10:
        print(f"  ... and {len(failed_pdfs) - 10} more")

# Save processing report
report = {
    "processing_date": datetime.now().isoformat(),
    "total_files": len(all_pdf_paths),
    "successful_files": success_count,
    "failed_files": len(failed_pdfs),
    "failed_list": failed_pdfs,
    "success_rate": (success_count/len(all_pdf_paths))*100
}

with open(os.path.join(PROCESSED_DIR, "processing_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\n💾 Processing report saved to: {os.path.join(PROCESSED_DIR, 'processing_report.json')}")

# Optional: Clean up temporary directory
try:
    if os.path.exists(TEMP_UNZIP_DIR):
        shutil.rmtree(TEMP_UNZIP_DIR)
        print(f"🧹 Temporary directory cleaned: {TEMP_UNZIP_DIR}")
except Exception as e:
    print(f"⚠️  Could not clean temporary directory: {e}")

print(f"\n🎉 Processing completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

🔍 Scanning for PDF files...
📁 Found 318 PDF files for processing

🚀 Starting PDF processing at 2025-12-10 06:08:42

[1/318] Processing: Sri-Lanka-Law-Report-1979-3sihya.pdf
  ✓ Text extracted via PyMuPDF (1040454 chars)
  ✅ Successfully processed and saved: Sri-Lanka-Law-Report-1979-3sihya.json
  📊 Stats: 1023147 chars, 7265 sentences, Year: 19, Court: Supreme Court

[2/318] Processing: Sri-Lanka-Law-Report-1978-tldat3.pdf
  ✓ Text extracted via PyMuPDF (1172968 chars)
  ✅ Successfully processed and saved: Sri-Lanka-Law-Report-1978-tldat3.json
  📊 Stats: 1156545 chars, 8998 sentences, Year: 19, Court: Supreme Court

[3/318] Processing: Sri-Lanka-Law-Report-1980-a16ecz.pdf
  ✓ Text extracted via PyMuPDF (982113 chars)
  ✅ Successfully processed and saved: Sri-Lanka-Law-Report-1980-a16ecz.json
  📊 Stats: 968082 chars, 6994 sentences, Year: 19, Court: Supreme Court

[4/318] Processing: SLR-2005-Vol-1-vzfajl.pdf
  ✓ Text extracted via PyMuPDF (925769 chars)
  ✅ Successfully processed and s

Restore above code run again

In [10]:
print("🔄 RESTORING COMPLETE DATASET")
print("=" * 60)

# First, let's find ALL PDF files across your entire data directory
def find_all_pdf_files():
    """Find all PDF files in all subdirectories"""
    all_pdfs = []
    base_dir = "/content/drive/MyDrive/ai-legal-summarizer-data"

    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith('.pdf'):
                full_path = os.path.join(root, file)
                all_pdfs.append(full_path)

    return all_pdfs

# Get all PDF files
all_pdf_files = find_all_pdf_files()
print(f"🔍 Found {len(all_pdf_files)} PDF files total")

# Check which ones are already processed
processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
processed_names = [f.replace('.json', '.pdf') for f in processed_files]

print(f"📊 Already processed: {len(processed_files)} files")
print(f"📋 Need to process: {len(all_pdf_files) - len(processed_files)} files")

# Identify missing files
missing_files = []
for pdf_path in all_pdf_files:
    pdf_name = os.path.basename(pdf_path)
    if pdf_name not in processed_names:
        missing_files.append(pdf_path)

print(f"🔄 Files to process: {len(missing_files)}")

# Process only the missing files
if missing_files:
    print(f"\n🚀 Processing {len(missing_files)} missing files...")

    success_count = 0
    for i, file_path in enumerate(missing_files):
        file_name = os.path.basename(file_path)
        print(f"\n[{i+1}/{len(missing_files)}] Processing: {file_name}")

        # Validate PDF
        if not validate_pdf(file_path):
            print(f"  ❌ Invalid PDF file")
            continue

        # Check file size
        file_size = os.path.getsize(file_path)
        if file_size < 1000:
            print(f"  ⚠️  File too small ({file_size} bytes)")
            continue

        try:
            # Extract text
            raw_text = robust_extract_text_from_pdf(file_path)

            if not raw_text or len(raw_text.strip()) < 100:
                print(f"  ❌ Insufficient text extracted")
                continue

            # Process text
            cleaned_text = clean_text(raw_text)
            sentences = segment_sentences(cleaned_text)

            # Extract metadata
            case_year = extract_case_year(cleaned_text)
            case_number = extract_case_number(cleaned_text)
            court = extract_court(cleaned_text)

            # Create entry
            entry = {
                "file_name": file_name,
                "file_size_bytes": file_size,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text,
                "sentences": sentences,
                "case_year": case_year,
                "case_number": case_number,
                "court": court,
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned_text),
                "sentence_count": len(sentences)
            }

            # Save JSON
            json_filename = file_name.replace(".pdf", ".json")
            save_path = os.path.join(PROCESSED_DIR, json_filename)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            success_count += 1
            print(f"  ✅ Processed and saved: {json_filename}")

        except Exception as e:
            print(f"  ❌ Error: {str(e)}")

    print(f"\n🎉 Restored {success_count} missing files")
else:
    print("✅ All files are already processed!")

# Final count
final_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
print(f"\n📁 Final total: {len(final_files)} processed files")

🔄 RESTORING COMPLETE DATASET
🔍 Found 86 PDF files total
📊 Already processed: 321 files
📋 Need to process: -235 files
🔄 Files to process: 0
✅ All files are already processed!

📁 Final total: 321 processed files


For Failed files

In [12]:
!apt-get update
!apt-get install -y poppler-utils

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,519 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/p

In [14]:
print("🔬 LAUNCHING DEEP FORENSIC RECOVERY")
print("=" * 60)

def deep_forensic_recovery(file_path):
    """Deep forensic analysis and recovery of corrupted PDFs"""
    file_name = os.path.basename(file_path)
    print(f"\n🔬 DEEP FORENSIC: {file_name}")
    print(f"   📏 File size: {os.path.getsize(file_path):,} bytes")

    try:
        # Read entire file as binary
        with open(file_path, 'rb') as f:
            full_content = f.read()

        # Analyze file structure
        print(f"   📊 File analysis:")
        print(f"      - PDF header: {full_content[:20]}")
        print(f"      - PDF trailer: {full_content[-50:] if len(full_content) > 50 else 'TOO SHORT'}")
        print(f"      - Contains 'stream': {b'stream' in full_content}")
        print(f"      - Contains 'endstream': {b'endstream' in full_content}")

        # Method 1: Extract any readable text from binary
        print(f"   🔍 Scanning for embedded text...")

        # Look for text between parentheses (common in PDF text objects)
        text_objects = re.findall(b'\(([^)]+)\)', full_content)
        recovered_text = ""

        for text_obj in text_objects:
            try:
                decoded = text_obj.decode('utf-8', errors='ignore')
                if len(decoded.strip()) > 10:  # Meaningful text
                    recovered_text += decoded + " "
            except:
                pass

        if recovered_text:
            print(f"   ✅ Found {len(recovered_text)} chars in text objects")
            return recovered_text

        # Method 2: Look for stream data that might contain text
        print(f"   🔍 Scanning stream data...")
        stream_pattern = re.findall(b'stream[\r\n]+(.*?)[\r\n]+endstream', full_content, re.DOTALL)

        for stream in stream_pattern:
            try:
                # Try to decompress or decode stream data
                if len(stream) > 100:
                    # Look for ASCII text in stream
                    text_in_stream = re.findall(b'[A-Za-z0-9\s.,;:!?]{20,}', stream)
                    if text_in_stream:
                        stream_text = b' '.join(text_in_stream).decode('utf-8', errors='ignore')
                        recovered_text += stream_text + " "
            except:
                pass

        if recovered_text:
            print(f"   ✅ Found {len(recovered_text)} chars in streams")
            return recovered_text

        # Method 3: Bruteforce text extraction from entire binary
        print(f"   🔍 Bruteforce text scanning...")
        # Look for consecutive printable ASCII characters
        all_text = re.findall(b'[\\x20-\\x7E]{10,}', full_content)
        if all_text:
            bruteforce_text = b' '.join(all_text).decode('ascii', errors='ignore')
            if len(bruteforce_text) > 100:
                print(f"   ✅ Found {len(bruteforce_text)} chars via bruteforce")
                return bruteforce_text

        # Method 4: Try to repair the PDF structure
        print(f"   🔧 Attempting structural repair...")
        repaired_content = attempt_structural_repair(full_content)
        if repaired_content:
            try:
                # Try to extract from repaired content
                import fitz
                with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as temp_file:
                    temp_file.write(repaired_content)
                    temp_path = temp_file.name

                doc = fitz.open(temp_path)
                repaired_text = ""
                for page in doc:
                    repaired_text += page.get_text()
                doc.close()
                os.unlink(temp_path)

                if repaired_text.strip():
                    print(f"   ✅ Structural repair successful: {len(repaired_text)} chars")
                    return repaired_text
            except:
                if temp_path and os.path.exists(temp_path):
                    os.unlink(temp_path)

        print(f"   ❌ No recoverable text found")
        return None

    except Exception as e:
        print(f"   ❌ Forensic analysis failed: {e}")
        return None

def attempt_structural_repair(content):
    """Attempt to repair PDF structure"""
    try:
        # Ensure proper header
        if not content.startswith(b'%PDF'):
            content = b'%PDF-1.4\n' + content

        # Ensure proper trailer
        if not content.endswith(b'%%EOF\n'):
            content = content + b'\n%%EOF\n'

        # Try to fix truncated objects
        # Look for incomplete streams and close them
        if b'stream' in content and b'endstream' not in content:
            # Add missing endstream
            content = content + b'\nendstream\n'

        return content
    except:
        return None

# Final recovery attempt
failed_files = [
    'New-Law-Report-Vol-46.pdf',
    'New-Law-Report-Vol-44.pdf',
    'New-Law-Report-Vol-40.pdf',
    'New-Law-Report-Vol-45.pdf',
    'New-Law-Report-Vol-47.pdf'
]

forensic_recovered = 0

for failed_file in failed_files:
    file_path = os.path.join(RAW_DIR, failed_file)

    raw_text = deep_forensic_recovery(file_path)

    if raw_text and len(raw_text.strip()) > 50:
        try:
            # Basic cleaning
            cleaned_text = clean_text(raw_text)

            # Create minimal entry
            entry = {
                "file_name": failed_file,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text,
                "sentences": segment_sentences(cleaned_text),
                "case_year": extract_case_year(cleaned_text),
                "case_number": extract_case_number(cleaned_text),
                "court": extract_court(cleaned_text),
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned_text),
                "sentence_count": len(segment_sentences(cleaned_text)),
                "recovery_status": "FORENSIC_RECOVERED_PARTIAL"
            }

            json_filename = failed_file.replace(".pdf", ".json")
            save_path = os.path.join(PROCESSED_DIR, json_filename)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            forensic_recovered += 1
            print(f"🎉 FORENSIC RECOVERY: {failed_file} → {len(cleaned_text)} chars")

        except Exception as e:
            print(f"❌ Failed to save recovered data: {e}")
    else:
        print(f"💀 FORENSIC RECOVERY FAILED: {failed_file}")

print(f"\n🔬 FORENSIC RECOVERY COMPLETE")
print(f"✅ Recovered: {forensic_recovered}/5 files")
print(f"🎯 Overall success: {313 + forensic_recovered}/318 files ({((313 + forensic_recovered)/318)*100:.1f}%)")

if forensic_recovered == 0:
    print(f"\n💡 RECOMMENDATION:")
    print(f"These 5 files are severely corrupted and cannot be recovered.")
    print(f"Your 313 successful files (98.4%) provide an excellent dataset!")
    print(f"Consider finding alternative sources for volumes 40, 44-47 if needed.")

<>:26: SyntaxWarning: invalid escape sequence '\('
<>:50: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\('
<>:50: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3109387361.py:26: SyntaxWarning: invalid escape sequence '\('
  text_objects = re.findall(b'\(([^)]+)\)', full_content)
/tmp/ipython-input-3109387361.py:50: SyntaxWarning: invalid escape sequence '\s'
  text_in_stream = re.findall(b'[A-Za-z0-9\s.,;:!?]{20,}', stream)


🔬 LAUNCHING DEEP FORENSIC RECOVERY

🔬 DEEP FORENSIC: New-Law-Report-Vol-46.pdf
   📏 File size: 4,194,304 bytes
   📊 File analysis:
      - PDF header: b'%PDF-1.4\r\n%\xe2\xe3\xcf\xd3\r\n1 0'
      - PDF trailer: b'1\r\n/Subtype /TrueType\r\n/ToUnicode 2581 0 R\r\n/Type '
      - Contains 'stream': True
      - Contains 'endstream': True
   🔍 Scanning for embedded text...
   ✅ Found 1143723 chars in text objects
🎉 FORENSIC RECOVERY: New-Law-Report-Vol-46.pdf → 1130020 chars

🔬 DEEP FORENSIC: New-Law-Report-Vol-44.pdf
   📏 File size: 8,388,608 bytes
   📊 File analysis:
      - PDF header: b'%PDF-1.4\r\n%\xe2\xe3\xcf\xd3\r\n1 0'
      - PDF trailer: b'FontDescriptor\r\n>>\r\nendobj\r\n3615 0 obj\r\n<<\r\n/Filte'
      - Contains 'stream': True
      - Contains 'endstream': True
   🔍 Scanning for embedded text...
   ✅ Found 2421818 chars in text objects
🎉 FORENSIC RECOVERY: New-Law-Report-Vol-44.pdf → 2390368 chars

🔬 DEEP FORENSIC: New-Law-Report-Vol-40.pdf
   📏 File size: 32,505,856 byte

In [15]:
print("\n🔧 SPECIAL RECOVERY FOR PROBLEMATIC FILES")
print("=" * 50)

problematic_files = [
    'New-Law-Report-Vol-40.pdf',
    'New-Law-Report-Vol-44.pdf',
    'New-Law-Report-Vol-45.pdf',
    'New-Law-Report-Vol-46.pdf',
    'New-Law-Report-Vol-47.pdf'
]

for problem_file in problematic_files:
    # Find the file
    file_path = None
    for root, dirs, files in os.walk("/content/drive/MyDrive/ai-legal-summarizer-data"):
        if problem_file in files:
            file_path = os.path.join(root, problem_file)
            break

    if file_path and os.path.exists(file_path):
        print(f"\n🔍 Processing problematic file: {problem_file}")

        # Use the forensic recovery approach
        raw_text = deep_forensic_recovery(file_path)

        if raw_text and len(raw_text.strip()) > 100:
            cleaned_text = clean_text(raw_text)
            sentences = segment_sentences(cleaned_text)

            entry = {
                "file_name": problem_file,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text,
                "sentences": sentences,
                "case_year": extract_case_year(cleaned_text),
                "case_number": extract_case_number(cleaned_text),
                "court": extract_court(cleaned_text),
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned_text),
                "sentence_count": len(sentences),
                "recovery_status": "FORENSIC_RECOVERED_PARTIAL"
            }

            json_filename = problem_file.replace(".pdf", ".json")
            save_path = os.path.join(PROCESSED_DIR, json_filename)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            print(f"  ✅ Forensic recovery successful: {len(cleaned_text)} chars")
        else:
            print(f"  ❌ Forensic recovery failed")
    else:
        print(f"  ❌ File not found: {problem_file}")


🔧 SPECIAL RECOVERY FOR PROBLEMATIC FILES

🔍 Processing problematic file: New-Law-Report-Vol-40.pdf

🔬 DEEP FORENSIC: New-Law-Report-Vol-40.pdf
   📏 File size: 32,505,856 bytes
   📊 File analysis:
      - PDF header: b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n688 0'
      - PDF trailer: b"uCj\x8bc\x1d\xa02\xb5E\x99\x14&\xe3LXQ\x1fK\x0c\xc1\x19U5B\xdf\xfb\xa6\xee\xc8\xcf\xab\x06\xfc\x85\xd2\xa7\x87\x10\x18L\xc4\xa8a\xee\xee\xd3\x89\xc4'"
      - Contains 'stream': True
      - Contains 'endstream': True
   🔍 Scanning for embedded text...
   ✅ Found 9075012 chars in text objects
  ✅ Forensic recovery successful: 9011684 chars

🔍 Processing problematic file: New-Law-Report-Vol-44.pdf

🔬 DEEP FORENSIC: New-Law-Report-Vol-44.pdf
   📏 File size: 8,388,608 bytes
   📊 File analysis:
      - PDF header: b'%PDF-1.4\r\n%\xe2\xe3\xcf\xd3\r\n1 0'
      - PDF trailer: b'FontDescriptor\r\n>>\r\nendobj\r\n3615 0 obj\r\n<<\r\n/Filte'
      - Contains 'stream': True
      - Contains 'endstream': True
   🔍 Scanning 

In [16]:
# Quick verification of your complete dataset
import os
import json

processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
print(f"📁 Total processed files: {len(processed_files)}")

# Sample some recovered files to verify quality
recovered_files = [f for f in processed_files if 'Vol-4' in f]
for json_file in recovered_files[:2]:
    json_path = os.path.join(PROCESSED_DIR, json_file)
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        print(f"\n📄 {data['file_name']}")
        print(f"   Status: {data.get('recovery_status', 'ORIGINAL')}")
        print(f"   Text: {data['text_length']:,} chars")
        print(f"   Year: {data['case_year']} | Court: {data['court']}")
        if 'recovery_status' in data:
            print(f"   🎯 FORENSIC RECOVERY SUCCESS!")

📁 Total processed files: 321

📄 New-Law-Report-Vol-41.pdf
   Status: ORIGINAL
   Text: 1,852,071 chars
   Year: 20 | Court: Supreme Court

📄 New-Law-Report-Vol-42.pdf
   Status: ORIGINAL
   Text: 1,823,779 chars
   Year: 20 | Court: Supreme Court


In [18]:
import os
import json

def verify_recovered_quality():
    recovered_count = 0
    total_chars = 0
    corrupted_files = []

    # Filter out non-case files (like combined_legal_cases.json, processing_report.json, glossary files)
    all_json_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
    processed_files = [f for f in all_json_files if not any(x in f.lower() for x in ['report', 'combined', 'glossary'])]

    for json_file in processed_files:
        json_path = os.path.join(PROCESSED_DIR, json_file)
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                # Try to read and parse the JSON
                content = f.read().strip()
                if not content:  # Empty file
                    print(f"⚠️  Empty file: {json_file}")
                    corrupted_files.append(json_file)
                    continue

                data = json.loads(content)

            # This check is for individual case files
            if data.get('recovery_status') == 'FORENSIC_RECOVERED_PARTIAL':
                recovered_count += 1
                total_chars += data.get('text_length', 0)

        except json.JSONDecodeError as e:
            print(f"❌ Corrupted JSON: {json_file} - {e}")
            corrupted_files.append(json_file)
        except Exception as e:
            print(f"❌ Error reading {json_file}: {e}")
            corrupted_files.append(json_file)

    print(f"\n🔍 Quality Report:")
    print(f"   Recovered files: {recovered_count}")
    if recovered_count > 0:
        print(f"   Total recovered text: {total_chars:,} characters")
        print(f"   Average per file: {total_chars//recovered_count:,} characters")
    else:
        print(f"   No recovered files found")

    if corrupted_files:
        print(f"\n❌ Corrupted files found: {len(corrupted_files)}")
        for corrupted in corrupted_files[:5]:  # Show first 5
            print(f"   - {corrupted}")

    return corrupted_files

# Run the fixed verification
corrupted_files = verify_recovered_quality()


🔍 Quality Report:
   Recovered files: 0
   No recovered files found


In [19]:
def repair_corrupted_files(corrupted_files):
    """Repair or reprocess corrupted JSON files"""
    print(f"\n🔧 REPAIRING {len(corrupted_files)} CORRUPTED FILES")
    print("=" * 50)

    repaired_count = 0
    for corrupted_file in corrupted_files:
        pdf_filename = corrupted_file.replace('.json', '.pdf')
        print(f"\n🔧 Repairing: {pdf_filename}")

        # Find the original PDF file
        pdf_path = None
        for root, dirs, files in os.walk("/content/drive/MyDrive/ai-legal-summarizer-data"):
            if pdf_filename in files:
                pdf_path = os.path.join(root, pdf_filename)
                break

        if not pdf_path or not os.path.exists(pdf_path):
            print(f"   ❌ Original PDF not found: {pdf_filename}")
            continue

        try:
            # Reprocess the file
            if not validate_pdf(pdf_path):
                print(f"   ❌ Invalid PDF")
                continue

            file_size = os.path.getsize(pdf_path)
            if file_size < 1000:
                print(f"   ❌ File too small")
                continue

            # Extract text
            raw_text = robust_extract_text_from_pdf(pdf_path)

            if not raw_text or len(raw_text.strip()) < 100:
                print(f"   ❌ Insufficient text")
                continue

            # Process text
            cleaned_text = clean_text(raw_text)
            sentences = segment_sentences(cleaned_text)

            # Check if this is one of the problematic files that needs forensic recovery
            recovery_status = None
            if pdf_filename in ['New-Law-Report-Vol-40.pdf', 'New-Law-Report-Vol-44.pdf',
                              'New-Law-Report-Vol-45.pdf', 'New-Law-Report-Vol-46.pdf',
                              'New-Law-Report-Vol-47.pdf']:
                recovery_status = "FORENSIC_RECOVERED_PARTIAL"

            # Create new entry
            entry = {
                "file_name": pdf_filename,
                "file_size_bytes": file_size,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text,
                "sentences": sentences,
                "case_year": extract_case_year(cleaned_text),
                "case_number": extract_case_number(cleaned_text),
                "court": extract_court(cleaned_text),
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned_text),
                "sentence_count": len(sentences)
            }

            if recovery_status:
                entry["recovery_status"] = recovery_status

            # Save the repaired JSON
            json_path = os.path.join(PROCESSED_DIR, corrupted_file)
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            repaired_count += 1
            print(f"   ✅ Repaired: {len(cleaned_text)} chars")

        except Exception as e:
            print(f"   ❌ Repair failed: {e}")

    print(f"\n🎉 Repaired {repaired_count}/{len(corrupted_files)} files")
    return repaired_count

# Repair corrupted files
if corrupted_files:
    repaired_count = repair_corrupted_files(corrupted_files)

    # Verify again after repair
    print(f"\n🔍 VERIFYING AFTER REPAIR")
    verify_recovered_quality()
else:
    print("✅ No corrupted files found!")

✅ No corrupted files found!


In [20]:
def comprehensive_quality_check():
    """Comprehensive check of all processed files"""
    print(f"\n📊 COMPREHENSIVE DATASET QUALITY CHECK")
    print("=" * 50)

    processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
    valid_files = 0
    total_chars = 0
    file_sizes = []

    for json_file in processed_files:
        json_path = os.path.join(PROCESSED_DIR, json_file)
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if data.get('text_length', 0) > 100:  # Valid file
                valid_files += 1
                total_chars += data['text_length']
                file_sizes.append(data['text_length'])

        except:
            pass  # Skip corrupted files

    print(f"📁 Total JSON files: {len(processed_files)}")
    print(f"✅ Valid files (100+ chars): {valid_files}")
    print(f"📊 Total characters: {total_chars:,}")
    if valid_files > 0:
        print(f"📈 Average text per file: {total_chars//valid_files:,} chars")
        print(f"📏 Size range: {min(file_sizes):,} - {max(file_sizes):,} chars")

    # Check forensic recovery
    forensic_files = []
    for json_file in processed_files:
        json_path = os.path.join(PROCESSED_DIR, json_file)
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if data.get('recovery_status') == 'FORENSIC_RECOVERED_PARTIAL':
                    forensic_files.append(json_file)
        except:
            pass

    print(f"🔧 Forensic recovered files: {len(forensic_files)}")
    for f_file in forensic_files:
        print(f"   - {f_file}")

# Run comprehensive check
comprehensive_quality_check()


📊 COMPREHENSIVE DATASET QUALITY CHECK
📁 Total JSON files: 321
✅ Valid files (100+ chars): 318
📊 Total characters: 188,151,575
📈 Average text per file: 591,671 chars
📏 Size range: 3,224 - 9,011,684 chars
🔧 Forensic recovered files: 5
   - New-Law-Report-Vol-46.json
   - New-Law-Report-Vol-44.json
   - New-Law-Report-Vol-47.json
   - New-Law-Report-Vol-45.json
   - New-Law-Report-Vol-40.json


In [21]:
def extract_text_from_pdf(path):
    text = ""

    # Try pdfplumber
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                t = p.extract_text()
                if t:
                    text += t + "\n"
    except:
        pass

    # Fallback: OCR
    if len(text.strip()) < 100:
        print("Using OCR for:", path)
        try:
            images = convert_from_path(path)
            for img in images:
                text += pytesseract.image_to_string(img)
        except:
            print("OCR failed for:", path)

    return text

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install Dependencies

In [23]:
!pip install pdfplumber pytesseract pillow nltk pdf2image
!apt-get install poppler-utils -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 72 not upgraded.


Import Libraries

In [24]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path, convert_from_bytes
from PIL import Image
import os, re, json
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Extract text with OCR fallback

In [25]:
def extract_text_from_pdf(path):
    text = ""

    # Try pdfplumber
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                t = p.extract_text()
                if t:
                    text += t + "\n"
    except:
        pass

    # Fallback: OCR
    if len(text.strip()) < 100:
        print("Using OCR for:", path)
        try:
            images = convert_from_path(path)
            for img in images:
                text += pytesseract.image_to_string(img)
        except:
            print("OCR failed for:", path)

    return text

Metadata Extractors (Sri Lanka–specific)

In [26]:
def extract_case_year(text):
    m = re.search(r"(19|20)\d{2}", text)
    return int(m.group()) if m else None

def extract_case_number(text):
    m = re.search(r"(SC|CA|HC)\s+\w+/\d{2,4}", text)
    return m.group() if m else None

def extract_court(text):
    courts = ["Supreme Court", "Court of Appeal", "High Court"]
    for c in courts:
        if c.lower() in text.lower():
            return c
    return None

Cleaner + Sentence Splitter

In [27]:
from nltk.tokenize import sent_tokenize

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def segment_sentences(text):
    return sent_tokenize(text)

Preprocess ALL PDFs in Raw Folder

In [29]:
# Instead of reprocessing, use the COMPLETE dataset we already created
import os
import json

# Check what we actually have
processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
print(f"🎉 ALREADY PROCESSED: {len(processed_files)} legal cases")
print(f"📁 Location: {PROCESSED_DIR}")

# Load your complete dataset with error handling
dataset = []
corrupted_files = []
valid_files = 0

# Filter out non-case files (processing reports, combined files, etc.)
case_files_to_load = [f for f in processed_files if not any(x in f.lower() for x in ['report', 'combined', 'glossary'])]

for json_file in case_files_to_load:
    json_path = os.path.join(PROCESSED_DIR, json_file)
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Check if essential keys exist
        if 'file_name' in data and 'cleaned_text' in data:
            dataset.append(data)
            valid_files += 1
        else:
            print(f"⚠️ Missing essential keys: {json_file}")
            corrupted_files.append(json_file)

    except json.JSONDecodeError as e:
        print(f"❌ Corrupted JSON: {json_file} - {e}")
        corrupted_files.append(json_file)
    except Exception as e:
        print(f"❌ Error reading {json_file}: {e}")
        corrupted_files.append(json_file)

print(f"\n📊 DATASET STATUS:")
print(f"✅ Valid files: {valid_files}/{len(case_files_to_load)}")
print(f"❌ Corrupted files: {len(corrupted_files)}")
print(f"📚 Loaded {len(dataset)} cases into memory")
print(f"📊 Total characters: {sum(d.get('text_length', 0) for d in dataset):,}")

# Verify we have all volumes (only for valid files)
if dataset:
    volumes = [d['file_name'] for d in dataset if 'file_name' in d and 'Vol-4' in d['file_name']]
    print(f"\n✅ Recovered volumes 40,44-47: {len(volumes)} files")
    for vol in sorted(volumes):
        print(f"   📄 {vol}")
else:
    print("❌ No valid data found in dataset")

# Show sample of what we have
if dataset:
    print(f"\n🔍 SAMPLE DATA:")
    sample = dataset[0]
    print(f"File: {sample.get('file_name', 'N/A')}")
    print(f"Text length: {sample.get('text_length', len(sample.get('cleaned_text', '')))}")
    print(f"Year: {sample.get('case_year', 'N/A')}")
    print(f"Court: {sample.get('court', 'N/A')}")

🎉 ALREADY PROCESSED: 321 legal cases
📁 Location: /content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons

📊 DATASET STATUS:
✅ Valid files: 72/72
❌ Corrupted files: 0
📚 Loaded 72 cases into memory
📊 Total characters: 8,262,803

✅ Recovered volumes 40,44-47: 0 files

🔍 SAMPLE DATA:
File: 2006-2-SRI-L.-R.-Part-03-uckrf5.pdf
Text length: 61636
Year: 20
Court: Supreme Court


In [30]:
# Instead of reprocessing, use the COMPLETE dataset we already created
import os
import json

# Check what we actually have
processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json')]
print(f"📁 Total JSON files: {len(processed_files)}")

# Filter out non-case files (processing reports, combined files, etc.)
case_files = [f for f in processed_files if not any(x in f.lower() for x in ['report', 'combined', 'glossary'])]
non_case_files = [f for f in processed_files if f not in case_files]

print(f"📚 Legal case files: {len(case_files)}")
print(f"📊 Support files: {len(non_case_files)}")
if non_case_files:
    print(f"   Support files: {non_case_files}")

# Load only the legal case files
dataset = []
corrupted_files = []

for json_file in case_files:
    json_path = os.path.join(PROCESSED_DIR, json_file)
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Check if this is a valid legal case
        if 'file_name' in data and 'cleaned_text' in data:
            dataset.append(data)
        else:
            print(f"⚠️ Missing essential keys: {json_file}")
            corrupted_files.append(json_file)

    except json.JSONDecodeError as e:
        print(f"❌ Corrupted JSON: {json_file} - {e}")
        corrupted_files.append(json_file)
    except Exception as e:
        print(f"❌ Error reading {json_file}: {e}")
        corrupted_files.append(json_file)

print(f"\n📊 DATASET STATUS:")
print(f"✅ Valid case files: {len(dataset)}")
print(f"❌ Corrupted case files: {len(corrupted_files)}")
print(f"📊 Total characters: {sum(d.get('text_length', 0) for d in dataset):,}")

# Verify we have all volumes
if dataset:
    volumes = [d['file_name'] for d in dataset if 'Vol-4' in d['file_name']]
    print(f"\n✅ Recovered volumes 40,44-47: {len(volumes)} files")
    for vol in sorted(volumes):
        print(f"   📄 {vol}")

    # Show overall statistics
    print(f"\n📈 DATASET OVERVIEW:")
    print(f"   Total cases: {len(dataset)}")
    print(f"   Total text: {sum(d.get('text_length', 0) for d in dataset):,} chars")
    print(f"   Average per case: {sum(d.get('text_length', 0) for d in dataset)//len(dataset):,} chars")

    # Show years covered
    years = [d.get('case_year') for d in dataset if d.get('case_year')]
    if years:
        print(f"   Years covered: {min(years)} - {max(years)}")

    # Show courts represented
    courts = [d.get('court') for d in dataset if d.get('court')]
    unique_courts = set(courts)
    print(f"   Courts: {len(unique_courts)} different courts")

📁 Total JSON files: 321
📚 Legal case files: 72
📊 Support files: 249
   Support files: ['New-Law-Report-Vol-53.json', 'New-Law-Report-Vol-54.json', 'New-Law-Report-Vol-56.json', 'New-Law-Report-Vol-55.json', 'New-Law-Report-Vol-58.json', 'New-Law-Report-Vol-57.json', 'New-Law-Report-Vol-60.json', 'New-Law-Report-Vol-59.json', 'New-Law-Report-Vol-62.json', 'New-Law-Report-Vol-63.json', 'New-Law-Report-Vol-64.json', 'New-Law-Report-Vol-61.json', 'New-Law-Report-Vol-65.json', 'New-Law-Report-Vol-66.json', 'New-Law-Report-Vol-69.json', 'New-Law-Report-Vol-67.json', 'New-Law-Report-Vol-70.json', 'New-Law-Report-Vol-72.json', 'New-Law-Report-Vol-71.json', 'New-Law-Report-Vol-75.json', 'New-Law-Report-Vol-74.json', 'New-Law-Report-Vol-73.json', 'New-Law-Report-Vol-77.json', 'New-Law-Report-Vol-76.json', 'New-Law-Report-Vol-78.json', 'New-Law-Report-Vol-79-1.json', 'New-Law-Report-Vol-79-2.json', 'New-Law-Report-Vol-80.json', 'New-Law-Report-V4-lwkd8c.json', 'New-Law-Report-V3-iwdkvy.json', 'Ne

Save Combined Training Dataset (Optional)

In [31]:
# Create combined dataset only from valid case files
if dataset:
    print(f"\n🔄 CREATING COMBINED LEGAL DATASET")

    combined_data = {
        "metadata": {
            "total_cases": len(dataset),
            "total_characters": sum(d.get('text_length', 0) for d in dataset),
            "creation_date": datetime.now().isoformat(),
            "description": "Sri Lanka Legal Cases Dataset",
            "years_covered": f"{min([d.get('case_year', 1900) for d in dataset if d.get('case_year')])}-{max([d.get('case_year', 2024) for d in dataset if d.get('case_year')])}",
            "source": "Sri Lanka Law Reports"
        },
        "cases": dataset
    }

    combined_path = os.path.join(PROCESSED_DIR, "combined_legal_cases.json")
    with open(combined_path, "w", encoding="utf-8") as f:
        json.dump(combined_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Combined dataset saved: {combined_path}")
    print(f"   Contains {len(dataset)} legal cases")
    print(f"   Total: {sum(d.get('text_length', 0) for d in dataset):,} characters")

else:
    print("❌ No valid case files found to combine")


🔄 CREATING COMBINED LEGAL DATASET
✅ Combined dataset saved: /content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons/combined_legal_cases.json
   Contains 72 legal cases
   Total: 8,262,803 characters


Export Sinhala/Tamil Glossary (Required by Backend)

In [32]:
# Create comprehensive legal glossary
print(f"\n📚 CREATING LEGAL GLOSSARY")

# Make sure CORPUS_DIR exists
CORPUS_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

glossary = {
    "fundamental_rights": {
        "10": {"en": "Freedom of thought, conscience and religion", "si": "චින්තනය, හෘද සාක්ෂිය හා ආගමේ නිදහස", "ta": "சிந்தனை, மனசாட்சி மற்றும் மத சுதந்திரம்"},
        "11": {"en": "Freedom from torture", "si": "පීඩනයෙන් නිදහස", "ta": "வதைத்தலில் இருந்து சுதந்திரம்"},
        "12": {"en": "Right to equality", "si": "සමානාත්මතාවයේ අයිතිය", "ta": "சமத்துவ உரிமை"},
        "13": {"en": "Freedom from arbitrary arrest, detention and punishment", "si": "අත්තනෝමතික අත්අඩංගුව, රඳවා තබාගැනීම් සහ දඬුවම් වලින් නිදහස", "ta": "தன்னிச்சையான கைது, தடுப்பு மற்றும் தண்டனையிலிருந்து சுதந்திரம்"},
        "14": {"en": "Freedom of speech, assembly, association, occupation", "si": "කථනය, සමුළුව, සංගම්, වෘත්තිය නිදහස", "ta": "பேச்சு, கூட்டம், சங்கம், தொழில் சுதந்திரம்"},
    },
    "legal_terms": {
        "mens rea": {"en": "criminal intention", "si": "අපරාධක මනස", "ta": "குற்றமுள்ள மனநிலை"},
        "actus reus": {"en": "criminal act", "si": "අපරාධක ක්‍රියාව", "ta": "குற்றச் செயல்"},
        "res judicata": {"en": "matter already judged", "si": "දැනටමත් තීන්දු කළ කාරණය", "ta": "முடிவு செய்யப்பட்ட விஷயம்"},
        "stare decisis": {"en": "to stand by things decided", "si": "තීරණය කරන ලද දේවල් මත සිටීම", "ta": "முன்பு முடிவு செய்ததைப் பின்பற்றுதல்"},
    }
}

glossary_path = os.path.join(CORPUS_DIR, "legal_glossary_si_en_ta.json")
with open(glossary_path, "w", encoding="utf-8") as f:
    json.dump(glossary, f, indent=2, ensure_ascii=False)

print(f"✅ Legal glossary saved: {glossary_path}")

print(f"\n🎉 LEGAL AI FOUNDATION COMPLETE!")
print("=" * 50)
print(f"📚 Cases: {len(dataset)} legal decisions")
print(f"📖 Glossary: Multilingual legal terms")
print(f"💾 Combined dataset ready for AI training")
print("=" * 50)


📚 CREATING LEGAL GLOSSARY
✅ Legal glossary saved: /content/drive/MyDrive/ai-legal-summarizer-data/corpus/legal_glossary_si_en_ta.json

🎉 LEGAL AI FOUNDATION COMPLETE!
📚 Cases: 72 legal decisions
📖 Glossary: Multilingual legal terms
💾 Combined dataset ready for AI training


## Verify Complete Dataset (ZIP files included)

In [33]:
import os
import json

# Define the paths
PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons"
COMBINED_FILE = "/content/drive/MyDrive/ai-legal-summarizer-data/processed/combined_legal_cases.json"

print("🔍 VERIFYING COMPLETE DATASET")
print("=" * 60)

# Check individual JSON files
if os.path.exists(PROCESSED_DIR):
    json_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.json') and 'report' not in f.lower() and 'combined' not in f.lower()]
    print(f"\n📁 Individual JSON files: {len(json_files)}")

    # Count files from ZIP archives (1981-2014)
    zip_sourced = [f for f in json_files if any(year in f for year in ['1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990',
                                                                         '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000',
                                                                         '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
                                                                         '2011', '2012', '2013', '2014'])]

    print(f"   ✅ From ZIP files (1981-2014): {len(zip_sourced)} cases")
    print(f"   📄 From regular PDFs: {len(json_files) - len(zip_sourced)} cases")

    # Show sample of ZIP-sourced files
    if zip_sourced:
        print(f"\n📋 Sample ZIP-sourced files:")
        for sample in sorted(zip_sourced)[:5]:
            print(f"   - {sample}")
else:
    print(f"❌ Directory not found: {PROCESSED_DIR}")

# Check combined file
print(f"\n📊 CHECKING COMBINED DATASET FILE")
if os.path.exists(COMBINED_FILE):
    with open(COMBINED_FILE, 'r', encoding='utf-8') as f:
        combined_data = json.load(f)

    metadata = combined_data.get('metadata', {})
    cases = combined_data.get('cases', [])

    print(f"✅ Combined file exists: {COMBINED_FILE}")
    print(f"\n📈 Metadata:")
    print(f"   Total cases: {metadata.get('total_cases', len(cases))}")
    print(f"   Total characters: {metadata.get('total_characters', 0):,}")
    print(f"   Years covered: {metadata.get('years_covered', 'N/A')}")
    print(f"   Creation date: {metadata.get('creation_date', 'N/A')}")

    print(f"\n📚 Actual cases in file: {len(cases)}")

    # Analyze years covered
    years = [case.get('case_year') for case in cases if case.get('case_year')]
    if years:
        year_counts = {}
        for year in years:
            year_counts[year] = year_counts.get(year, 0) + 1

        print(f"\n📅 Cases by decade:")
        decades = {}
        for year, count in year_counts.items():
            decade = (year // 10) * 10
            decades[decade] = decades.get(decade, 0) + count

        for decade in sorted(decades.keys()):
            print(f"   {decade}s: {decades[decade]} cases")

    # Check for ZIP-sourced content
    zip_cases = [case for case in cases if any(year in case.get('file_name', '')
                 for year in ['1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990',
                             '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000',
                             '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
                             '2011', '2012', '2013', '2014'])]

    print(f"\n🗜️ ZIP-sourced cases in combined file: {len(zip_cases)}/{len(cases)}")

    # File size
    file_size = os.path.getsize(COMBINED_FILE)
    print(f"\n💾 Combined file size: {file_size:,} bytes ({file_size / (1024*1024):.2f} MB)")

    # Show sample cases
    print(f"\n📄 Sample cases from combined file:")
    for i, case in enumerate(cases[:3]):
        print(f"\n   Case {i+1}:")
        print(f"      File: {case.get('file_name', 'N/A')}")
        print(f"      Year: {case.get('case_year', 'N/A')}")
        print(f"      Court: {case.get('court', 'N/A')}")
        print(f"      Text length: {case.get('text_length', 0):,} chars")
        print(f"      Sentences: {case.get('sentence_count', 0)}")

else:
    print(f"❌ Combined file NOT FOUND: {COMBINED_FILE}")
    print(f"\n💡 The combined file needs to be created from individual JSONs")

print(f"\n{'=' * 60}")
print(f"🎯 CONCLUSION:")
if os.path.exists(COMBINED_FILE):
    with open(COMBINED_FILE, 'r', encoding='utf-8') as f:
        combined_data = json.load(f)
        total = len(combined_data.get('cases', []))

    if total > 100:
        print(f"✅ Complete dataset exists with {total} cases (ZIP files INCLUDED)")
        print(f"📥 This file is ready to download to your local machine")
    elif total == 72:
        print(f"⚠️  Combined file has only 72 cases (ZIP files NOT INCLUDED)")
        print(f"🔄 Need to regenerate combined file with all processed JSONs")
    else:
        print(f"📊 Combined file has {total} cases")
else:
    print(f"⚠️  Combined file doesn't exist - needs to be created")
print(f"{'=' * 60}")

🔍 VERIFYING COMPLETE DATASET

📁 Individual JSON files: 72
   ✅ From ZIP files (1981-2014): 72 cases
   📄 From regular PDFs: 0 cases

📋 Sample ZIP-sourced files:
   - 2005-2-SRI-L.-R.-Part-01-04-9yo1uv.json
   - 2005-2-SRI-L.-R.-Part-05-3szc9q.json
   - 2005-2-SRI-L.-R.-Part-06-atvhnb.json
   - 2005-2-SRI-L.-R.-Part-07-iokcdn.json
   - 2005-2-SRI-L.-R.-Part-08-n2ebjk.json

📊 CHECKING COMBINED DATASET FILE
✅ Combined file exists: /content/drive/MyDrive/ai-legal-summarizer-data/processed/combined_legal_cases.json

📈 Metadata:
   Total cases: 72
   Total characters: 8,262,803
   Years covered: 20-20
   Creation date: 2025-12-10T05:51:04.989358

📚 Actual cases in file: 72

📅 Cases by decade:
   20s: 72 cases

🗜️ ZIP-sourced cases in combined file: 72/72

💾 Combined file size: 25,743,409 bytes (24.55 MB)

📄 Sample cases from combined file:

   Case 1:
      File: 2006-2-SRI-L.-R.-Part-03-uckrf5.pdf
      Year: 20
      Court: Supreme Court
      Text length: 61,636 chars
      Sentences: 455

## Process ALL Remaining PDFs (Regular PDFs + ZIP files)

In [34]:
import os
import json
from datetime import datetime

print("🚀 PROCESSING ALL REMAINING PDFs")
print("=" * 60)

# Use the paths from your earlier cells
RAW_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/raw_pdfs"
TEMP_UNZIP_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs"
PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons"

# Create directories if needed
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Step 1: Extract ZIP files if not already done
print("\n📦 Step 1: Extracting ZIP files...")
if os.path.exists(RAW_DIR):
    zip_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith(".zip")]

    if zip_files:
        import zipfile
        os.makedirs(TEMP_UNZIP_DIR, exist_ok=True)

        print(f"Found {len(zip_files)} ZIP files")
        for zip_file_name in zip_files:
            zip_file_path = os.path.join(RAW_DIR, zip_file_name)
            extraction_folder = os.path.join(TEMP_UNZIP_DIR, os.path.splitext(zip_file_name)[0])

            if not os.path.exists(extraction_folder):
                try:
                    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                        os.makedirs(extraction_folder, exist_ok=True)
                        zip_ref.extractall(extraction_folder)
                    print(f"  ✅ Extracted: {zip_file_name}")
                except Exception as e:
                    print(f"  ❌ Failed: {zip_file_name} - {e}")
            else:
                print(f"  ⏭️  Already extracted: {zip_file_name}")
    else:
        print("  No ZIP files found")

# Step 2: Find ALL PDF files (from raw dir + unzipped)
print("\n📁 Step 2: Scanning for ALL PDF files...")
all_pdfs = []

# PDFs directly in raw directory
if os.path.exists(RAW_DIR):
    for f in os.listdir(RAW_DIR):
        if f.lower().endswith('.pdf'):
            all_pdfs.append(os.path.join(RAW_DIR, f))

# PDFs from extracted ZIP files
if os.path.exists(TEMP_UNZIP_DIR):
    for root, dirs, files in os.walk(TEMP_UNZIP_DIR):
        for f in files:
            if f.lower().endswith('.pdf'):
                all_pdfs.append(os.path.join(root, f))

print(f"  Found {len(all_pdfs)} total PDF files")

# Step 3: Check which are already processed
already_processed = []
if os.path.exists(PROCESSED_DIR):
    already_processed = [f.replace('.json', '.pdf') for f in os.listdir(PROCESSED_DIR)
                        if f.endswith('.json')]

print(f"  Already processed: {len(already_processed)} files")

# Step 4: Find unprocessed files
to_process = []
for pdf_path in all_pdfs:
    pdf_name = os.path.basename(pdf_path)
    if pdf_name not in already_processed:
        to_process.append(pdf_path)

print(f"  Need to process: {len(to_process)} files")

# Step 5: Process the remaining files
if to_process:
    print(f"\n⚙️  Step 3: Processing {len(to_process)} unprocessed PDFs...")
    print("This will take some time...\n")

    success = 0
    failed = 0

    for i, pdf_path in enumerate(to_process):
        pdf_name = os.path.basename(pdf_path)
        print(f"[{i+1}/{len(to_process)}] {pdf_name}... ", end='')

        try:
            # Validate PDF
            if not validate_pdf(pdf_path):
                print("❌ Invalid PDF")
                failed += 1
                continue

            file_size = os.path.getsize(pdf_path)
            if file_size < 1000:
                print("❌ Too small")
                failed += 1
                continue

            # Extract text
            raw_text = robust_extract_text_from_pdf(pdf_path)

            if not raw_text or len(raw_text.strip()) < 100:
                print(f"❌ No text ({len(raw_text.strip())} chars)")
                failed += 1
                continue

            # Process
            cleaned_text = clean_text(raw_text)
            sentences = segment_sentences(cleaned_text)

            # Create entry
            entry = {
                "file_name": pdf_name,
                "file_size_bytes": file_size,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text,
                "sentences": sentences,
                "case_year": extract_case_year(cleaned_text),
                "case_number": extract_case_number(cleaned_text),
                "court": extract_court(cleaned_text),
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned_text),
                "sentence_count": len(sentences)
            }

            # Save
            json_filename = pdf_name.replace(".pdf", ".json")
            save_path = os.path.join(PROCESSED_DIR, json_filename)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            print(f"✅ {len(cleaned_text):,} chars")
            success += 1

        except Exception as e:
            print(f"❌ Error: {str(e)[:50]}")
            failed += 1

    print(f"\n{'=' * 60}")
    print(f"📊 PROCESSING COMPLETE")
    print(f"  ✅ Successfully processed: {success}")
    print(f"  ❌ Failed: {failed}")
    print(f"  📁 Total processed files: {len(already_processed) + success}")
else:
    print(f"\n✅ All PDF files are already processed!")
    print(f"  Total: {len(already_processed)} files")

print(f"\n{'=' * 60}")

🚀 PROCESSING ALL REMAINING PDFs

📦 Step 1: Extracting ZIP files...
Found 41 ZIP files
  ✅ Extracted: Sri-Lanka-Law-Report-1982--01146027300.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1981--01152412000.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1983--01135193700.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1984--01112298900.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1989--01147314000.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1986--01108935900.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1988--01111016900.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1985--01142351800.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1987--01100085300.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1991--01133613100.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1990--01144709700.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1993--01142872400.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1992--01102465400.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1995--01120546400.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1996--01112887100.zip
  ✅ Extracted: Sri-Lanka-Law-Report-1998--01

## Create Combined Dataset with ALL Cases

In [42]:
import os
import json
from datetime import datetime

print("📦 CREATING COMBINED DATASET")
print("=" * 60)

PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed"
OUTPUT_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load all JSON files
print("\n📁 Loading all processed cases...")
dataset = []
corrupted = []

if os.path.exists(PROCESSED_DIR):
    json_files = [f for f in os.listdir(PROCESSED_DIR)
                  if f.endswith('.json')
                  and 'processing_report' not in f.lower()
                  and 'combined' not in f.lower()
                  and 'glossary' not in f.lower()]

    print(f"  Found {len(json_files)} JSON files")

    for json_file in json_files:
        json_path = os.path.join(PROCESSED_DIR, json_file)
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Validate essential fields
            if 'file_name' in data and 'cleaned_text' in data:
                dataset.append(data)
            else:
                print(f"  ⚠️  Missing fields: {json_file}")
                corrupted.append(json_file)
        except Exception as e:
            print(f"  ❌ Error loading {json_file}: {e}")
            corrupted.append(json_file)

    print(f"  ✅ Loaded: {len(dataset)} valid cases")
    if corrupted:
        print(f"  ❌ Corrupted: {len(corrupted)} files")
else:
    print(f"  ❌ Directory not found: {PROCESSED_DIR}")

# Create combined file
if dataset:
    print(f"\n📊 Creating combined dataset...")

    # Calculate metadata
    years = [d.get('case_year') for d in dataset if d.get('case_year')]
    min_year = min(years) if years else None
    max_year = max(years) if years else None

    combined_data = {
        "metadata": {
            "total_cases": len(dataset),
            "total_characters": sum(d.get('text_length', 0) for d in dataset),
            "creation_date": datetime.now().isoformat(),
            "description": "Sri Lanka Legal Cases Dataset - Complete Collection",
            "years_covered": f"{min_year}-{max_year}" if min_year and max_year else "Unknown",
            "source": "Sri Lanka Law Reports (ZIP files + Regular PDFs)",
            "processing_method": "Multi-method PDF extraction with OCR fallback"
        },
        "cases": dataset
    }

    # Save combined file
    output_path = os.path.join(OUTPUT_DIR, "combined_legal_cases.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(combined_data, f, indent=2, ensure_ascii=False)

    file_size = os.path.getsize(output_path)

    print(f"\n✅ COMBINED DATASET CREATED!")
    print(f"  📍 Location: {output_path}")
    print(f"  📊 Total cases: {len(dataset)}")
    print(f"  📝 Total characters: {sum(d.get('text_length', 0) for d in dataset):,}")
    print(f"  📅 Years: {min_year} - {max_year}")
    print(f"  💾 File size: {file_size:,} bytes ({file_size / (1024*1024):.2f} MB)")

    # Show statistics
    print(f"\n📈 Dataset Statistics:")

    # By decade
    if years:
        decades = {}
        for year in years:
            decade = (year // 10) * 10
            decades[decade] = decades.get(decade, 0) + 1

        print(f"  Cases by decade:")
        for decade in sorted(decades.keys()):
            print(f"    {decade}s: {decades[decade]} cases")

    # By court
    courts = [d.get('court') for d in dataset if d.get('court')]
    court_counts = {}
    for court in courts:
        court_counts[court] = court_counts.get(court, 0) + 1

    if court_counts:
        print(f"\n  Cases by court:")
        for court, count in sorted(court_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {court}: {count} cases")

    print(f"\n{'=' * 60}")
    print(f"🎉 READY FOR DOWNLOAD!")
    print(f"📥 Download this file to your local machine:")
    print(f"   {output_path}")
    print(f"📂 Then place it at:")
    print(f"   e:\\ai-legal-summarizer\\data\\processed\\combined_legal_cases.json")
    print(f"{'=' * 60}")
else:
    print("\n❌ No valid cases found to combine")

📦 CREATING COMBINED DATASET

📁 Loading all processed cases...
  Found 318 JSON files
  ✅ Loaded: 318 valid cases

📊 Creating combined dataset...

✅ COMBINED DATASET CREATED!
  📍 Location: /content/drive/MyDrive/ai-legal-summarizer-data/processed/combined_legal_cases.json
  📊 Total cases: 318
  📝 Total characters: 164,442,926
  📅 Years: 1978 - 2076
  💾 File size: 517,213,398 bytes (493.25 MB)

📈 Dataset Statistics:
  Cases by decade:
    1970s: 2 cases
    1980s: 21 cases
    1990s: 25 cases
    2000s: 39 cases
    2010s: 78 cases
    2020s: 75 cases
    2070s: 1 cases

  Cases by court:
    Supreme Court: 239 cases

🎉 READY FOR DOWNLOAD!
📥 Download this file to your local machine:
   /content/drive/MyDrive/ai-legal-summarizer-data/processed/combined_legal_cases.json
📂 Then place it at:
   e:\ai-legal-summarizer\data\processed\combined_legal_cases.json


## Debug: Find ALL Processed Files

In [36]:
import os

print("🔍 FINDING ALL PROCESSED JSON FILES")
print("=" * 60)

base_dir = "/content/drive/MyDrive/ai-legal-summarizer-data"

# Search for all JSON files
all_json_locations = {}

for root, dirs, files in os.walk(base_dir):
    json_files = [f for f in files if f.endswith('.json') and 'report' not in f.lower()]
    if json_files:
        all_json_locations[root] = len(json_files)

if all_json_locations:
    print("\n📂 Directories with JSON files:")
    for location, count in sorted(all_json_locations.items(), key=lambda x: x[1], reverse=True):
        print(f"\n  📁 {location}")
        print(f"     Count: {count} files")

        # Show first few files
        try:
            files = [f for f in os.listdir(location) if f.endswith('.json') and 'report' not in f.lower()]
            print(f"     Sample: {', '.join(files[:3])}")
        except:
            pass

    print(f"\n{'=' * 60}")
    print(f"📊 TOTAL JSON FILES: {sum(all_json_locations.values())}")

    # Identify the correct directory
    if all_json_locations:
        main_dir = max(all_json_locations.items(), key=lambda x: x[1])[0]
        print(f"\n✅ MAIN DIRECTORY (most files):")
        print(f"   {main_dir}")
        print(f"\n💡 This should be used as PROCESSED_DIR")
else:
    print("\n❌ No JSON files found!")

print(f"{'=' * 60}")

🔍 FINDING ALL PROCESSED JSON FILES

📂 Directories with JSON files:

  📁 /content/drive/MyDrive/ai-legal-summarizer-data/processed
     Count: 74 files
     Sample: combined.json, SLR-2005-Vol-1-vzfajl.json, SLR-2006-Vol-3-ba9o2y.json

  📁 /content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons
     Count: 74 files
     Sample: 2006-2-SRI-L.-R.-Part-03-uckrf5.json, SLR-2006-Vol-3-ba9o2y.json, SLR-2005-Vol-1-vzfajl.json

  📁 /content/drive/MyDrive/ai-legal-summarizer-data/sri_lanka_legal_corpus
     Count: 1 files
     Sample: legal_glossary_si_en_ta.json

  📁 /content/drive/MyDrive/ai-legal-summarizer-data/corpus
     Count: 1 files
     Sample: legal_glossary_si_en_ta.json

📊 TOTAL JSON FILES: 150

✅ MAIN DIRECTORY (most files):
   /content/drive/MyDrive/ai-legal-summarizer-data/processed

💡 This should be used as PROCESSED_DIR


## Consolidate & Process ALL PDFs (Fixed Paths)

In [37]:
import os
import json
import shutil
from datetime import datetime

print("🔧 CONSOLIDATING & PROCESSING ALL PDFs")
print("=" * 60)

# Define paths
RAW_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/raw_pdfs"
TEMP_UNZIP_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/temp_unzipped_pdfs"
MAIN_PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed"
ALT_PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed_jsons"

os.makedirs(MAIN_PROCESSED_DIR, exist_ok=True)

# Step 1: Consolidate files from both directories
print("\n📦 Step 1: Consolidating processed files...")
all_processed_files = set()

# Get files from both directories
for dir_path in [MAIN_PROCESSED_DIR, ALT_PROCESSED_DIR]:
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith('.json') and 'combined' not in f.lower() and 'report' not in f.lower() and 'glossary' not in f.lower()]
        print(f"  📁 {os.path.basename(dir_path)}: {len(files)} files")

        # Copy files to main directory if they're in alt directory
        if dir_path == ALT_PROCESSED_DIR:
            for file in files:
                src = os.path.join(ALT_PROCESSED_DIR, file)
                dst = os.path.join(MAIN_PROCESSED_DIR, file)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    print(f"    ↪️  Copied: {file}")

        all_processed_files.update(files)

print(f"  ✅ Total unique processed files: {len(all_processed_files)}")

# Step 2: Find ALL PDFs
print("\n📁 Step 2: Finding ALL PDF files...")
all_pdfs = {}

# PDFs in raw directory
if os.path.exists(RAW_DIR):
    for f in os.listdir(RAW_DIR):
        if f.lower().endswith('.pdf'):
            all_pdfs[f] = os.path.join(RAW_DIR, f)

# PDFs from ZIP files
if os.path.exists(TEMP_UNZIP_DIR):
    for root, dirs, files in os.walk(TEMP_UNZIP_DIR):
        for f in files:
            if f.lower().endswith('.pdf'):
                # Use filename as key to avoid duplicates
                if f not in all_pdfs:
                    all_pdfs[f] = os.path.join(root, f)

print(f"  📄 Total unique PDF files: {len(all_pdfs)}")

# Step 3: Find unprocessed PDFs
print("\n🔍 Step 3: Identifying unprocessed PDFs...")
processed_pdf_names = {f.replace('.json', '.pdf') for f in all_processed_files}
to_process = {name: path for name, path in all_pdfs.items() if name not in processed_pdf_names}

print(f"  ✅ Already processed: {len(processed_pdf_names)} PDFs")
print(f"  🆕 Need to process: {len(to_process)} PDFs")

# Show sample of what needs processing
if to_process:
    print(f"\n  Sample files to process:")
    for i, name in enumerate(list(to_process.keys())[:5]):
        print(f"    - {name}")

# Step 4: Process remaining PDFs
if to_process:
    print(f"\n⚙️  Step 4: Processing {len(to_process)} unprocessed PDFs...")
    print(f"This may take 10-30 minutes depending on file sizes...\n")

    success = 0
    failed = 0
    failed_list = []

    for i, (pdf_name, pdf_path) in enumerate(to_process.items()):
        print(f"[{i+1}/{len(to_process)}] {pdf_name[:50]}... ", end='', flush=True)

        try:
            # Validate
            if not validate_pdf(pdf_path):
                print("❌ Invalid")
                failed += 1
                failed_list.append(pdf_name)
                continue

            file_size = os.path.getsize(pdf_path)
            if file_size < 1000:
                print("❌ Too small")
                failed += 1
                failed_list.append(pdf_name)
                continue

            # Extract
            raw_text = robust_extract_text_from_pdf(pdf_path)

            if not raw_text or len(raw_text.strip()) < 100:
                print(f"❌ No text")
                failed += 1
                failed_list.append(pdf_name)
                continue

            # Process
            cleaned = clean_text(raw_text)
            sentences = segment_sentences(cleaned)

            # Create entry
            entry = {
                "file_name": pdf_name,
                "file_size_bytes": file_size,
                "raw_text": raw_text,
                "cleaned_text": cleaned,
                "sentences": sentences,
                "case_year": extract_case_year(cleaned),
                "case_number": extract_case_number(cleaned),
                "court": extract_court(cleaned),
                "processing_timestamp": datetime.now().isoformat(),
                "text_length": len(cleaned),
                "sentence_count": len(sentences)
            }

            # Save
            json_name = pdf_name.replace(".pdf", ".json")
            save_path = os.path.join(MAIN_PROCESSED_DIR, json_name)

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(entry, f, indent=2, ensure_ascii=False)

            print(f"✅ {len(cleaned):,} chars")
            success += 1

        except Exception as e:
            print(f"❌ {str(e)[:30]}")
            failed += 1
            failed_list.append(pdf_name)

    print(f"\n{'=' * 60}")
    print(f"📊 PROCESSING COMPLETE!")
    print(f"  ✅ Success: {success}")
    print(f"  ❌ Failed: {failed}")
    if failed_list:
        print(f"\n  Failed files:")
        for f in failed_list[:10]:
            print(f"    - {f}")
        if len(failed_list) > 10:
            print(f"    ... and {len(failed_list) - 10} more")
else:
    print(f"\n✅ All PDFs already processed!")

# Final count
final_count = len([f for f in os.listdir(MAIN_PROCESSED_DIR) if f.endswith('.json') and 'combined' not in f.lower() and 'report' not in f.lower()])
print(f"\n{'=' * 60}")
print(f"🎉 TOTAL PROCESSED: {final_count} legal cases")
print(f"📂 Location: {MAIN_PROCESSED_DIR}")
print(f"{'=' * 60}")

🔧 CONSOLIDATING & PROCESSING ALL PDFs

📦 Step 1: Consolidating processed files...
  📁 processed: 72 files
  📁 processed_jsons: 72 files
  ✅ Total unique processed files: 72

📁 Step 2: Finding ALL PDF files...
  📄 Total unique PDF files: 318

🔍 Step 3: Identifying unprocessed PDFs...
  ✅ Already processed: 72 PDFs
  🆕 Need to process: 246 PDFs

  Sample files to process:
    - Sri-Lanka-Law-Report-1979-3sihya.pdf
    - Sri-Lanka-Law-Report-1978-tldat3.pdf
    - Sri-Lanka-Law-Report-1980-a16ecz.pdf
    - New-Law-Report-Vol-43 (1).pdf
    - New-Law-Report-Vol-48.pdf

⚙️  Step 4: Processing 246 unprocessed PDFs...
This may take 10-30 minutes depending on file sizes...

[1/246] Sri-Lanka-Law-Report-1979-3sihya.pdf...   ✓ Text extracted via PyMuPDF (1040454 chars)
✅ 1,023,147 chars
[2/246] Sri-Lanka-Law-Report-1978-tldat3.pdf...   ✓ Text extracted via PyMuPDF (1172968 chars)
✅ 1,156,545 chars
[3/246] Sri-Lanka-Law-Report-1980-a16ecz.pdf...   ✓ Text extracted via PyMuPDF (982113 chars)
✅ 968,

## Download Combined Dataset


In [44]:
from google.colab import files
import os

# Path to the combined dataset
combined_file = "/content/drive/MyDrive/ai-legal-summarizer-data/processed/combined_legal_cases.json"

# Check if file exists
if os.path.exists(combined_file):
    file_size_mb = os.path.getsize(combined_file) / (1024 * 1024)
    print(f"📥 Downloading combined_legal_cases.json")
    print(f"📊 File size: {file_size_mb:.2f} MB")
    print(f"⏳ Please wait, this may take a few minutes...")

    files.download(combined_file)

    print(f"\n✅ Download complete!")
    print(f"\n📍 Place the file at:")
    print(f"   e:\\ai-legal-summarizer\\data\\processed\\combined_legal_cases.json")
else:
    print(f"❌ File not found: {combined_file}")
    print(f"⚠️  Please run the 'Create Combined Dataset' cell first (around line 1508)")

📥 Downloading combined_legal_cases.json
📊 File size: 493.25 MB
⏳ Please wait, this may take a few minutes...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!

📍 Place the file at:
   e:\ai-legal-summarizer\data\processed\combined_legal_cases.json


## Verify All Processed Files

In [43]:
import os

MAIN_PROCESSED_DIR = "/content/drive/MyDrive/ai-legal-summarizer-data/processed"

print("🔍 CHECKING PROCESSED DIRECTORY")
print("=" * 60)

# Count all JSON files
all_json = [f for f in os.listdir(MAIN_PROCESSED_DIR) if f.endswith('.json')]
case_json = [f for f in all_json if 'combined' not in f.lower() and 'report' not in f.lower() and 'glossary' not in f.lower()]

print(f"📂 Directory: {MAIN_PROCESSED_DIR}")
print(f"📄 Total JSON files: {len(all_json)}")
print(f"📋 Case files (excluding combined/report/glossary): {len(case_json)}")

# Show breakdown
combined = [f for f in all_json if 'combined' in f.lower()]
reports = [f for f in all_json if 'report' in f.lower()]
glossary = [f for f in all_json if 'glossary' in f.lower()]

print(f"\n📊 Breakdown:")
print(f"  ✅ Case files: {len(case_json)}")
print(f"  📦 Combined files: {len(combined)}")
print(f"  📋 Report files: {len(reports)}")
print(f"  📚 Glossary files: {len(glossary)}")

print(f"\n📝 Sample case files (first 10):")
for i, f in enumerate(case_json[:10]):
    print(f"  {i+1}. {f}")

if len(case_json) != 313:
    print(f"\n⚠️  WARNING: Expected 313 case files, but found {len(case_json)}")
    print(f"  This means only {len(case_json)} were saved to this directory.")
else:
    print(f"\n✅ All 313 case files are present!")

print("=" * 60)

🔍 CHECKING PROCESSED DIRECTORY
📂 Directory: /content/drive/MyDrive/ai-legal-summarizer-data/processed
📄 Total JSON files: 320
📋 Case files (excluding combined/report/glossary): 72

📊 Breakdown:
  ✅ Case files: 72
  📦 Combined files: 2
  📋 Report files: 246
  📚 Glossary files: 0

📝 Sample case files (first 10):
  1. SLR-2005-Vol-1-vzfajl.json
  2. SLR-2006-Vol-3-ba9o2y.json
  3. 2006-2-SRI-L.-R.-Part-03-uckrf5.json
  4. 2006-1-SRI-L.-R.-Part-1-6-uyrtx8.json
  5. 2006-1-SRI-L.-R.-Part-7-rtjfwi.json
  6. 2006-1-SRI-L.-R.-Part-8-3kkndh.json
  7. 2006-1-SRI-L.-R.-Part-9-n1jnwd.json
  8. 2006-1-SRI-L.-R.-Part-10-6jmpgz.json
  9. 2006-1-SRI-L.-R.-Part-11-jccjmn.json
  10. 2006-1-SRI-L.-R.-Part-12-gktona.json

⚠️  WARNING: Expected 313 case files, but found 72
  This means only 72 were saved to this directory.
